# NorthStar Notebook 2: SQL within R

This notebook covers the **SQL within R** requirement.

It demonstrates:
- loading the dataset in R
- running SQL queries using `sqldf`
- filtering, grouping, joining and ordering
- interpreting SQL query outputs for NorthStar

In [17]:
install.packages("sqldf")
install.packages("readr")
install.packages("dplyr")
install.packages("lubridate")

library(sqldf)
library(readr)
library(dplyr)
library(lubridate)

print("Packages loaded")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



[1] "Packages loaded"


## 1. Upload and extract dataset

In [18]:
uploaded <- "/content/northstar_dataset.zip"
unzip(uploaded, exdir = "northstar_dataset_extracted")

all_files <- list.files("northstar_dataset_extracted", recursive = TRUE, full.names = TRUE)
base_path <- dirname(all_files[grepl("orders.csv", all_files)][1])
print(base_path)

[1] "northstar_dataset_extracted/northstar_dataset"


## 2. Load CSV files

In [19]:
orders <- read_csv(file.path(base_path, "orders.csv"), show_col_types = FALSE)
deliveries <- read_csv(file.path(base_path, "deliveries.csv"), show_col_types = FALSE)
customers <- read_csv(file.path(base_path, "customers.csv"), show_col_types = FALSE)
complaints <- read_csv(file.path(base_path, "complaints.csv"), show_col_types = FALSE)
drivers <- read_csv(file.path(base_path, "drivers.csv"), show_col_types = FALSE)
vehicles <- read_csv(file.path(base_path, "vehicles.csv"), show_col_types = FALSE)
hubs <- read_csv(file.path(base_path, "hubs.csv"), show_col_types = FALSE)
incidents <- read_csv(file.path(base_path, "incidents.csv"), show_col_types = FALSE)
app_events <- read_csv(file.path(base_path, "app_events.csv"), show_col_types = FALSE)

cat("orders:", nrow(orders), "\n")
cat("deliveries:", nrow(deliveries), "\n")
cat("complaints:", nrow(complaints), "\n")

orders: 1250 
deliveries: 950 
complaints: 320 


## 3. Query 1: Orders by service type

In [20]:
query1 <- "
SELECT service_type,
       COUNT(*) AS total_orders,
       ROUND(AVG(order_value), 2) AS average_order_value,
       ROUND(SUM(order_value), 2) AS total_order_value
FROM orders
GROUP BY service_type
ORDER BY total_orders DESC
"

orders_by_service <- sqldf(query1)
orders_by_service

service_type,total_orders,average_order_value,total_order_value
<chr>,<int>,<dbl>,<dbl>
Passenger,341,96.07,32761.11
Parcel,308,87.62,26985.62
Retail,297,90.01,26734.06
Business,165,92.25,15220.43
Medical,139,87.14,12111.93


**Interpretation:** This query shows which service types have the highest demand and revenue value. NorthStar can use this to plan driver, vehicle and hub resources.

## 4. Query 2: Delivery status count

In [21]:
query2 <- "
SELECT delivery_status,
       COUNT(*) AS number_of_deliveries
FROM deliveries
GROUP BY delivery_status
ORDER BY number_of_deliveries DESC
"

delivery_status_sql <- sqldf(query2)
delivery_status_sql

delivery_status,number_of_deliveries
<chr>,<int>
OnTime,616
Delayed,202
Failed,132


## 5. Query 3: Failed deliveries by dropoff zone

In [22]:
query3 <- "
SELECT o.dropoff_zone,
       COUNT(d.delivery_id) AS total_deliveries,
       SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
       ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(d.delivery_id), 2) AS failure_rate_percent,
       ROUND(AVG(d.customer_rating_post_delivery), 2) AS average_customer_rating
FROM deliveries d
JOIN orders o
  ON d.order_id = o.order_id
GROUP BY o.dropoff_zone
ORDER BY failure_rate_percent DESC
"

failed_by_zone <- sqldf(query3)
failed_by_zone

dropoff_zone,total_deliveries,failed_deliveries,failure_rate_percent,average_customer_rating
<chr>,<int>,<int>,<dbl>,<dbl>
Central,47,11,23.40,3.79
North,40,9,22.50,3.73
SOUTH,56,11,19.64,3.97
north,57,10,17.54,3.80
RiverSide,75,11,14.67,3.91
West,69,10,14.49,3.83
Ctr,49,7,14.29,3.73
CENTRAL,51,7,13.73,3.77
EAST,61,8,13.11,3.79


**Interpretation:** This query proves why integrated data is important. The delivery table is joined with the order table so that failure rate can be analysed by dropoff zone.

## 6. Query 4: Complaints by type and severity

In [23]:
query4 <- "
SELECT complaint_type,
       severity,
       COUNT(*) AS complaint_count,
       ROUND(AVG(resolution_days), 2) AS average_resolution_days,
       ROUND(AVG(compensation_amount), 2) AS average_compensation
FROM complaints
GROUP BY complaint_type, severity
ORDER BY complaint_count DESC
"

complaints_sql <- sqldf(query4)
complaints_sql

complaint_type,severity,complaint_count,average_resolution_days,average_compensation
<chr>,<chr>,<int>,<dbl>,<dbl>
Delay,Medium,56,5.96,18.21
MissedPickup,Medium,37,6.16,17.91
DriverBehaviour,Medium,31,5.42,15.88
Delay,Low,27,6.48,8.16
AppIssue,Medium,25,7.36,16.11
Delay,High,18,12.44,36.54
DriverBehaviour,High,16,13.75,38.39
MissedPickup,High,16,11.56,43.07
AppIssue,Low,15,6.07,13.25


## 7. Query 5: Driver performance

In [28]:
query5 <- "
SELECT dr.driver_id,
       dr.base_zone,
       dr.years_experience, -- Corrected from 'experience' to 'years_experience'
       dr.training_score,
       COUNT(de.delivery_id) AS total_deliveries,
       ROUND(AVG(de.customer_rating_post_delivery), 2) AS average_rating,
       ROUND(AVG(de.manual_route_override_count), 2) AS average_route_overrides,
       SUM(CASE WHEN de.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries
FROM drivers dr
LEFT JOIN deliveries de
  ON dr.driver_id = de.driver_id
GROUP BY dr.driver_id, dr.base_zone, dr.years_experience, dr.training_score
HAVING total_deliveries > 0
ORDER BY average_rating ASC
LIMIT 10
"

driver_performance <- sqldf(query5)
driver_performance

driver_id,base_zone,years_experience,training_score,total_deliveries,average_rating,average_route_overrides,failed_deliveries
<chr>,<chr>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<int>
D063,north,12,85.7,3,2.37,0.33,2
D111,Airport,15,79.2,4,2.64,0.50,2
D076,East,15,61.6,4,2.67,1.00,1
D141,North,14,71.4,9,2.93,0.78,2
D165,North,10,82.2,6,2.98,1.33,2
D021,South,7,91.8,2,3.10,2.50,0
D147,West,13,66.4,2,3.10,1.00,1
D091,Riverside,12,68.0,6,3.13,0.17,1
D053,West,5,83.3,7,3.14,0.86,1


In [26]:
colnames(drivers)

[1] "driver_id"        "base_zone"        "employment_type"  "years_experience"
[5] "training_score"   "driver_rating"    "shift_preference" "active_flag"

## 8. Query 6: Hub performance

In [29]:
query6 <- "
SELECT h.hub_id,
       h.hub_name,
       h.zone,
       h.capacity_score,
       COUNT(d.delivery_id) AS total_deliveries,
       SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
       ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(d.delivery_id), 2) AS failure_rate_percent,
       ROUND(AVG(d.fuel_or_charge_cost), 2) AS average_cost,
       ROUND(AVG(d.customer_rating_post_delivery), 2) AS average_rating
FROM hubs h
LEFT JOIN deliveries d
  ON h.hub_id = d.hub_id
GROUP BY h.hub_id, h.hub_name, h.zone, h.capacity_score
ORDER BY failure_rate_percent DESC
"

hub_performance <- sqldf(query6)
hub_performance

hub_id,hub_name,zone,capacity_score,total_deliveries,failed_deliveries,failure_rate_percent,average_cost,average_rating
<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>
H08,Midtown Relay,Central,63,128,26,20.31,11.71,3.88
H05,Central Core,Central,88,115,23,20.00,13.69,3.67
H06,Airport Hub,Airport,71,104,15,14.42,13.32,3.88
H04,West Gate,West,69,127,16,12.60,13.17,3.92
H01,North Exchange,North,82,136,17,12.50,12.76,3.84
H07,Riverside Hub,Riverside,66,115,14,12.17,12.92,3.88
H02,South Link,South,78,106,10,9.43,12.57,3.95
H03,East Dock,East,74,119,11,9.24,12.74,3.90


## 9. Query 7: Vehicle condition and delivery results

In [30]:
query7 <- "
SELECT v.vehicle_type,
       v.maintenance_status,
       COUNT(d.delivery_id) AS total_deliveries,
       ROUND(AVG(v.battery_health_pct), 2) AS average_battery_health,
       ROUND(AVG(v.odometer_km), 2) AS average_odometer,
       ROUND(AVG(d.customer_rating_post_delivery), 2) AS average_delivery_rating,
       SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries
FROM vehicles v
LEFT JOIN deliveries d
  ON v.vehicle_id = d.vehicle_id
GROUP BY v.vehicle_type, v.maintenance_status
ORDER BY failed_deliveries DESC
"

vehicle_sql <- sqldf(query7)
vehicle_sql

vehicle_type,maintenance_status,total_deliveries,average_battery_health,average_odometer,average_delivery_rating,failed_deliveries
<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<int>
CargoVan,InRepair,68,69.82,104131.01,3.46,22
Hybrid,InRepair,71,83.27,130176.85,3.78,21
Diesel,InRepair,55,75.65,93865.76,3.66,17
EV,InRepair,60,77.79,148940.78,3.64,17
Hybrid,Active,131,73.67,120494.78,3.94,15
CargoVan,Active,117,74.41,102701.86,3.92,11
EV,Active,214,83.00,119469.70,3.95,11
Diesel,Active,80,67.02,80029.27,4.03,8
CargoVan,Scheduled,38,68.96,140428.16,3.96,5


## 10. Export SQL outputs

In [31]:
dir.create("sql_in_r_outputs", showWarnings = FALSE)

write_csv(orders_by_service, "sql_in_r_outputs/orders_by_service.csv")
write_csv(delivery_status_sql, "sql_in_r_outputs/delivery_status_sql.csv")
write_csv(failed_by_zone, "sql_in_r_outputs/failed_by_zone.csv")
write_csv(complaints_sql, "sql_in_r_outputs/complaints_sql.csv")
write_csv(driver_performance, "sql_in_r_outputs/driver_performance.csv")
write_csv(hub_performance, "sql_in_r_outputs/hub_performance.csv")
write_csv(vehicle_sql, "sql_in_r_outputs/vehicle_sql.csv")

print(list.files("sql_in_r_outputs"))

[1] "complaints_sql.csv"      "delivery_status_sql.csv"
[3] "driver_performance.csv"  "failed_by_zone.csv"     
[5] "hub_performance.csv"     "orders_by_service.csv"  
[7] "vehicle_sql.csv"        
